# SoundStream Demo

This notebook demonstrates the main purpose of this repo - trained SoundStream neural audio codec. It takes any audio URL, passes it through the codec and plays the original vs reconstructed audio to understand quality of model

My implementation of this model has `STOI` = 0.8229399738928121 and `NISQA` = 1.9275813531784611 on `test-clean` of LibriSpeec.

## 1. Getting code

In [ ]:
!git clone https://github.com/Vdv09/Neural-Audio-Codec.git

%cd Neural-Audio-Codec

!pip install -r requirements.txt

## 2. Download best checkpoint

In [ ]:
import os

os.makedirs("checkpoints", exist_ok=True)

!gdown 13trOe0Fj-IwJxkHFumprQND0lOvKe9G5 -O checkpoints/model_best.pth

## 3. Load model

In [ ]:
import torch
from src.model import SoundStream

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

ckpt = torch.load("checkpoints/model_best.pth", map_location=device, weights_only=False)

model = SoundStream().to(device)
model.load_state_dict(ckpt["model"])
model.eval()

print(f"Loaded checkpoint from step {ckpt['step']}")

## 4. Model inference

Notice that you can use with any url for audio file

In [ ]:
import urllib.request
import torchaudio
import torch
import numpy as np
from IPython.display import display, Audio

SAMPLE_RATE = 16000

AUDIO_URL = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav"

urllib.request.urlretrieve(AUDIO_URL, "input.wav")

audio, sr = torchaudio.load("input.wav")

if sr != SAMPLE_RATE:
    audio = torchaudio.functional.resample(audio, sr, SAMPLE_RATE)

audio = audio[:1]

with torch.no_grad():
    out = model(audio.unsqueeze(0).to(device))

reconstructed = out["audio_hat"].squeeze().cpu()

print("Original:")
display(Audio(audio.numpy(), rate=SAMPLE_RATE))

print("Reconstructed:")
display(Audio(reconstructed.numpy(), rate=SAMPLE_RATE))

### Metrics

You can also make sure that our metrics are good

In [ ]:
!python evaluate.py \
  --checkpoint {path_to_checkpoint} \
  --data_dir {path_to_test_data}